In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

In [ ]:
import glob
import os
import time

import pandas as pd
import torch
import easyocr

# 스레드 수 고정
torch.set_num_threads(4)

# date_parser.py는 이 노트북과 같은 저장소 루트에 있어야 함
from date_parser import extract_expiry_fields

# [참가자 구현 영역]
image_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.*")))
image_ids = [os.path.splitext(os.path.basename(p))[0] for p in image_files]
print(f"입력 이미지 {len(image_files)}장")

# EasyOCR
reader = easyocr.Reader(
    ['ko', 'en'],
    gpu=False,
    model_storage_directory='./weights',
    download_enabled=False,
    verbose=False,
)

# 리사이즈 긴 변 기준(px)
# 잠정값 - YOLO 써보고 수정 필요
RESIZE_LONG_EDGE = 900

In [ ]:
import tempfile
from PIL import Image

# 리사이즈한 임시 파일을 둘 폴더
_resize_dir = tempfile.mkdtemp(prefix="itda_resized_")


def _resize_for_ocr(path, long_edge):
    """긴 변이 long_edge보다 크면 비율 유지해서 줄인다. 이미 작으면 원본 그대로."""
    with Image.open(path) as im:
        w, h = im.size
        if max(w, h) <= long_edge:
            return path
        scale = long_edge / max(w, h)
        im2 = im.convert("RGB").resize(
            (max(1, int(w * scale)), max(1, int(h * scale))), Image.LANCZOS
        )
        out_path = os.path.join(_resize_dir, os.path.basename(path))
        im2.save(out_path, quality=90)
        return out_path


results = []
t_start = time.time()
for img_id, path in zip(image_ids, image_files):
    try:
        ocr_path = _resize_for_ocr(path, RESIZE_LONG_EDGE)
        text = " ".join(reader.readtext(ocr_path, detail=0))
        parsed = extract_expiry_fields(text)
    except Exception as e:
        # 이미지 한 장이 깨져도 전체 실행이 죽어도 NONE 처리 후 계속 진행
        print(f"[WARN] {img_id} 처리 중 오류, NONE 처리: {e}")
        parsed = {"year": "NONE", "month": "NONE", "day": "NONE", "final_date": "NONE"}

    results.append({
        "image_id": img_id,
        "year": parsed["year"],
        "month": parsed["month"],
        "day": parsed["day"],
        "final_date": parsed["final_date"],
    })

elapsed = time.time() - t_start
n = max(1, len(results))
print(f"총 {len(results)}장 처리 완료, {elapsed:.1f}초 (장당 평균 {elapsed/n:.2f}초)")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")